# kael: quickstart

A deep learning framework written from scratch in C++, used here from Python.

In [ ]:
# Skip this cell if kael-tensor is already installed.
!pip install --quiet git+https://github.com/JetBerri/kael.cpp

In [ ]:
import numpy as np

import kael
from kael import nn, optim
from kael.nn import functional as F

kael.__version__

## Tensors

Factories accept shapes as arguments or as a tuple. `kael.tensor` infers shape and dtype from nested lists.

In [ ]:
print(kael.zeros(2, 3))
print(kael.ones((2, 2), dtype=kael.int64))
print(kael.tensor([[1.0, 2.0], [3.0, 4.0]]))

## Views share memory

Slicing, transposing and reshaping never copy: they only change shape, strides and offset.

In [ ]:
t = kael.arange(12, dtype=kael.float32).reshape(3, 4)
row = t[1]
row[0] = 100
print(t)
print(t.T.shape, t.T.stride(), t.T.is_contiguous())

## NumPy without copies

In [ ]:
array = np.ones(3)
view = kael.from_numpy(array)
array[0] = 7
print(view)
print(view.numpy())

## Autograd

In [ ]:
x = kael.tensor([1.0, 2.0, 3.0]).requires_grad_()
y = (x * x).sum()
y.backward()
print(y.grad_fn, x.grad)

## A short training run

`nn.Module` is plain Python, with the same shape as in PyTorch.

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(2, 16), nn.Tanh(), nn.Linear(16, 2))

    def forward(self, x):
        return self.layers(x)


kael.manual_seed(3)
inputs = kael.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
targets = kael.tensor([0, 1, 1, 0])
model = Net()
optimizer = optim.AdamW(model.parameters(), lr=0.05)
for step in range(301):
    optimizer.zero_grad()
    loss = F.cross_entropy(model(inputs), targets)
    loss.backward()
    optimizer.step()
    if step % 100 == 0:
        print(step, round(loss.item(), 4))
kael.argmax(model(inputs), 1)